# Proyecto 1 — Dashboard de Ventas y Logística
## Punto 7 — Modelado Relacional
**Autor:** Michael Hans Evans  
**Input:** `superstore_clean.csv` (9,993 filas, 25 columnas)  
**Output:** `superstore_model.db` (13 tablas, SQLite)  
**Modelo:** Snowflake Schema en 3FN — claves naturales + subrogadas para jerarquías  

---
> Este notebook construye el modelo relacional Snowflake de 13 tablas a partir  
> del dataset limpio. Cada dimensión está normalizada en 3FN eliminando dependencias  
> transitivas. Los resultados son validados contra los KPIs del dashboard Power BI.

## 0. Importación de librerías

In [1]:
import pandas as pd
import sqlite3
import os
from IPython.display import display

## 1. Carga del dataset limpio

In [2]:
DATA_URL = (
    "https://raw.githubusercontent.com/MHEvans02/"
    "Dashboard_Ventas_Logistica/main/data/superstore_clean.csv"
)
df = pd.read_csv(DATA_URL)

print(f"Filas: {len(df):,}  |  Columnas: {len(df.columns)}")
print(f"Período: {df['Order_Year'].min():.0f} — {df['Order_Year'].max():.0f}")

Filas: 9,993  |  Columnas: 25
Período: 2014 — 2017


## 2. Estrategia de modelado

### Tipo de modelo
**Snowflake Schema en Tercera Forma Normal (3FN)**  
Se eligió Snowflake sobre Star Schema para demostrar el dominio de modelado relacional clásico, eliminando todas las dependencias transitivas.

### Principios aplicados
- **1FN:** Atomicidad — cada columna tiene un único valor
- **2FN:** Sin dependencias parciales sobre claves compuestas  
- **3FN:** Sin dependencias transitivas — cada jerarquía separada en su propia tabla

### Decisiones clave de diseño
| Decisión | Justificación |
|---|---|
| `geography_key = City + Postal Code` | Resuelve ambigüedad de ciudades con mismo nombre en distintos estados |
| `order_product_key = Order ID + Product ID + Índice` | Garantiza unicidad cuando un producto aparece más de una vez en una orden |
| Claves subrogadas para jerarquías | `categories_key`, `state_key`, etc. — enteros livianos para JOINs eficientes |
| Claves naturales para entidades | `Customer ID`, `Product ID`, `Order ID` — ya son únicos en el dataset |

### Modelo resultante — 13 tablas
```
fact_orders (hechos)
├── dim_time                      → por Order ID
├── dim_customers                 → por Customer ID
│   └── dim_customers_segment     → por segment_key
├── dim_geography                 → por geography_key
│   ├── dim_geography_city        → por city_key
│   ├── dim_geography_state       → por state_key
│   └── dim_geography_region      → por region_key
├── dim_products_detail           → por Product ID
│   ├── dim_categories            → por categories_key
│   └── dim_sub_categories        → por sub_categories_key
├── dim_products                  → por Product ID
└── dim_shipment                  → por shipment_key
```

## 3. Construcción de tablas de dimensiones

### 3.1 dim_categories y dim_sub_categories

In [3]:
dim_categories = df[['Category']].drop_duplicates().reset_index(drop=True)
dim_categories.insert(0, 'categories_key', range(1, len(dim_categories)+1))

dim_sub_categories = df[['Sub-Category']].drop_duplicates().reset_index(drop=True)
dim_sub_categories.insert(0, 'sub_categories_key', range(1, len(dim_sub_categories)+1))

assert dim_categories['categories_key'].is_unique
assert dim_sub_categories['sub_categories_key'].is_unique

print(f"dim_categories: {len(dim_categories)} | dim_sub_categories: {len(dim_sub_categories)}")
display(dim_categories)
display(dim_sub_categories)

dim_categories: 3 | dim_sub_categories: 17


,categories_key,Category
0,1,Furniture
1,2,Office Supplies
2,3,Technology


,sub_categories_key,Sub-Category
0,1,Bookcases
1,2,Chairs
2,3,Labels
3,4,Tables
4,5,Storage
5,6,Furnishings
6,7,Art
7,8,Phones
8,9,Binders
9,10,Appliances


### 3.2 dim_products_detail y dim_products

In [4]:
# 3FN: Product Name NO va acá porque depende de Product ID, no de las claves de categoría
dim_products_detail = df[['Product ID','Category','Sub-Category']].drop_duplicates(subset=['Product ID']).reset_index(drop=True)
dim_products_detail = dim_products_detail.merge(dim_categories, on='Category', how='left')
dim_products_detail = dim_products_detail.merge(dim_sub_categories, on='Sub-Category', how='left')
dim_products_detail = dim_products_detail[['Product ID','categories_key','sub_categories_key']]

dim_products = df[['Product ID','Product Name']].drop_duplicates(subset=['Product ID']).reset_index(drop=True)

assert dim_products_detail['Product ID'].is_unique
assert dim_products['Product ID'].is_unique
assert dim_products_detail['categories_key'].isnull().sum() == 0
assert dim_products_detail['sub_categories_key'].isnull().sum() == 0

print(f"dim_products_detail: {len(dim_products_detail)} | dim_products: {len(dim_products)}")
display(dim_products_detail.head())
display(dim_products.head())

dim_products_detail: 1862 | dim_products: 1862


,Product ID,categories_key,sub_categories_key
0,FUR-BO-10001798,1,1
1,FUR-CH-10000454,1,2
2,OFF-LA-10000240,2,3
3,FUR-TA-10000577,1,4
4,OFF-ST-10000760,2,5


,Product ID,Product Name
0,FUR-BO-10001798,Bush Somerset Collection Bookcase
1,FUR-CH-10000454,"Hon Deluxe Fabric Upholstered Stacking Chairs,..."
2,OFF-LA-10000240,Self-Adhesive Address Labels for Typewriters b...
3,FUR-TA-10000577,Bretford CR4500 Series Slim Rectangular Table
4,OFF-ST-10000760,Eldon Fold 'N Roll Cart System


### 3.3 dim_customers y dim_customers_segment

In [5]:
# 3FN — Segment separado porque depende de segment_key, no de Customer ID
dim_customers_segment = df[['Segment']].drop_duplicates().reset_index(drop=True)
dim_customers_segment.insert(0, 'segment_key', range(1, len(dim_customers_segment)+1))

dim_customers = df[['Customer ID','Customer Name','Segment']].drop_duplicates(subset=['Customer ID']).reset_index(drop=True)
dim_customers = dim_customers.merge(dim_customers_segment, on='Segment', how='left')
dim_customers = dim_customers[['Customer ID','Customer Name','segment_key']]

assert dim_customers['Customer ID'].is_unique
assert dim_customers['segment_key'].isnull().sum() == 0

print(f"dim_customers_segment: {len(dim_customers_segment)} | dim_customers: {len(dim_customers)}")
display(dim_customers_segment)
display(dim_customers.head())

dim_customers_segment: 3 | dim_customers: 793


,segment_key,Segment
0,1,Consumer
1,2,Corporate
2,3,Home Office


,Customer ID,Customer Name,segment_key
0,CG-12520,Claire Gute,1
1,DV-13045,Darrin Van Huff,2
2,SO-20335,Sean O'Donnell,1
3,BH-11710,Brosina Hoffman,1
4,AA-10480,Andrew Allen,1


### 3.4 dim_geography (jerarquía City → State → Region)

In [6]:
# 3FN: City → State → Region son dependencias transitivas
dim_geography_region = df[['Region']].drop_duplicates().reset_index(drop=True)
dim_geography_region.insert(0, 'region_key', range(1, len(dim_geography_region)+1))

dim_geography_state = df[['State']].drop_duplicates().reset_index(drop=True)
dim_geography_state.insert(0, 'state_key', range(1, len(dim_geography_state)+1))

dim_geography_city = df[['City']].drop_duplicates().reset_index(drop=True)
dim_geography_city.insert(0, 'city_key', range(1, len(dim_geography_city)+1))

# geography_key = City + Postal Code (resuelve ciudades con mismo nombre en distintos estados)
dim_geography = df[['City','Postal Code','State','Region']].drop_duplicates(subset=['City','Postal Code']).reset_index(drop=True)
dim_geography['geography_key'] = dim_geography['City'].str.replace(' ','_') + '_' + dim_geography['Postal Code'].astype(str)
dim_geography = dim_geography.merge(dim_geography_city, on='City', how='left')
dim_geography = dim_geography.merge(dim_geography_state, on='State', how='left')
dim_geography = dim_geography.merge(dim_geography_region, on='Region', how='left')
dim_geography = dim_geography[['geography_key','Postal Code','city_key','state_key','region_key']]

assert dim_geography['geography_key'].is_unique
assert dim_geography.isnull().sum().sum() == 0

print(f"region: {len(dim_geography_region)} | state: {len(dim_geography_state)} | city: {len(dim_geography_city)} | geography: {len(dim_geography)}")
display(dim_geography.head())

region: 4 | state: 49 | city: 531 | geography: 632


,geography_key,Postal Code,city_key,state_key,region_key
0,Henderson_42420,42420,1,1,1
1,Los_Angeles_90036,90036,2,2,2
2,Fort_Lauderdale_33311,33311,3,3,1
3,Los_Angeles_90032,90032,2,2,2
4,Concord_28027,28027,4,4,1


### 3.5 dim_shipment

In [7]:
dim_shipment = df[['Ship Mode']].drop_duplicates().reset_index(drop=True)
dim_shipment.insert(0, 'shipment_key', range(1, len(dim_shipment)+1))

print(f"dim_shipment: {len(dim_shipment)}")
display(dim_shipment)

dim_shipment: 4


,shipment_key,Ship Mode
0,1,Second Class
1,2,Standard Class
2,3,First Class
3,4,Same Day


### 3.6 dim_time

In [8]:
# Se relaciona con fact_orders por Order ID, NO por fecha
# Esto evita duplicados ya que la misma orden puede tener varios productos
dim_time = (df[['Order ID','Order Date','Ship Date','Order_Year','Order_Month','Order_Quarter']]
    .drop_duplicates(subset=['Order ID'])
    .reset_index(drop=True))

assert dim_time['Order ID'].is_unique

print(f"dim_time: {len(dim_time)} órdenes únicas | {dim_time['Order Date'].min()} → {dim_time['Order Date'].max()}")
display(dim_time.head())

dim_time: 5009 órdenes únicas | 2014-01-03 → 2017-12-30


,Order ID,Order Date,Ship Date,Order_Year,Order_Month,Order_Quarter
0,CA-2016-152156,2016-11-08,2016-11-11,2016,11,4
1,CA-2016-138688,2016-06-12,2016-06-16,2016,6,2
2,US-2015-108966,2015-10-11,2015-10-18,2015,10,4
3,CA-2014-115812,2014-06-09,2014-06-14,2014,6,2
4,CA-2017-114412,2017-04-15,2017-04-20,2017,4,2


## 4. Construcción de fact_orders

In [9]:
df['geography_key'] = df['City'].str.replace(' ','_') + '_' + df['Postal Code'].astype(str)
df_fact = df.merge(dim_shipment, on='Ship Mode', how='left')

# Índice para resolver duplicados (misma orden, mismo producto)
df_fact = df_fact.reset_index(drop=True)
df_fact['Indice'] = df_fact.groupby(['Order ID','Product ID']).cumcount() + 1

df_fact['order_product_key'] = (df_fact['Order ID'] + '_' + 
                                 df_fact['Product ID'] + '_' + 
                                 df_fact['Indice'].astype(str))

fact_orders = df_fact[[
    'order_product_key', 'Order ID', 'Customer ID', 'Product ID',
    'geography_key', 'shipment_key', 'Indice',
    'Sales', 'Quantity', 'Discount', 'Discount_pct',
    'Profit', 'Profit_Margin_pct', 'Shipping_Days'
]].copy()

assert fact_orders['order_product_key'].is_unique
assert fact_orders['shipment_key'].isnull().sum() == 0

print(f"fact_orders: {len(fact_orders):,} filas × {len(fact_orders.columns)} columnas")
display(fact_orders.head())

fact_orders: 9,993 filas × 14 columnas


,order_product_key,Order ID,Customer ID,Product ID,geography_key,shipment_key,Indice,Sales,Quantity,Discount,Discount_pct,Profit,Profit_Margin_pct,Shipping_Days
0,CA-2016-152156_FUR-BO-10001798_1,CA-2016-152156,CG-12520,FUR-BO-10001798,Henderson_42420,1,1,261.9600,2,0.00,0.0,41.9136,16.00,3
1,CA-2016-152156_FUR-CH-10000454_1,CA-2016-152156,CG-12520,FUR-CH-10000454,Henderson_42420,1,1,731.9400,3,0.00,0.0,219.5820,30.00,3
2,CA-2016-138688_OFF-LA-10000240_1,CA-2016-138688,DV-13045,OFF-LA-10000240,Los_Angeles_90036,1,1,14.6200,2,0.00,0.0,6.8714,47.00,4
3,US-2015-108966_FUR-TA-10000577_1,US-2015-108966,SO-20335,FUR-TA-10000577,Fort_Lauderdale_33311,2,1,957.5775,5,0.45,45.0,-383.0310,-40.00,7
4,US-2015-108966_OFF-ST-10000760_1,US-2015-108966,SO-20335,OFF-ST-10000760,Fort_Lauderdale_33311,2,1,22.3680,2,0.20,20.0,2.5164,11.25,7


## 5. Resumen del modelo relacional

In [10]:
resumen = pd.DataFrame({
    'Tabla': [
        'dim_categories', 'dim_sub_categories', 'dim_customers_segment',
        'dim_customers', 'dim_geography_region', 'dim_geography_state',
        'dim_geography_city', 'dim_geography', 'dim_products_detail',
        'dim_products', 'dim_shipment', 'dim_time', 'fact_orders'
    ],
    'Filas': [
        len(dim_categories), len(dim_sub_categories), len(dim_customers_segment),
        len(dim_customers), len(dim_geography_region), len(dim_geography_state),
        len(dim_geography_city), len(dim_geography), len(dim_products_detail),
        len(dim_products), len(dim_shipment), len(dim_time), len(fact_orders)
    ],
    'Tipo': ['DIM']*12 + ['FACT']
}).set_index('Tabla')

print(f"Total tablas: {len(resumen)}")
resumen

Total tablas: 13


,Filas,Tipo
Tabla,,
dim_categories,3,DIM
dim_sub_categories,17,DIM
dim_customers_segment,3,DIM
dim_customers,793,DIM
dim_geography_region,4,DIM
dim_geography_state,49,DIM
dim_geography_city,531,DIM
dim_geography,632,DIM
dim_products_detail,1862,DIM


## 6. Exportación a SQLite

In [11]:
db_path = '../database/superstore_model.db'
if os.path.exists(db_path):
    os.remove(db_path)

conn = sqlite3.connect(db_path)

# Orden de exportación respeta dependencias FK
tablas = [
    ('dim_categories',        dim_categories),
    ('dim_sub_categories',    dim_sub_categories),
    ('dim_customers_segment', dim_customers_segment),
    ('dim_customers',         dim_customers),
    ('dim_geography_region',  dim_geography_region),
    ('dim_geography_state',   dim_geography_state),
    ('dim_geography_city',    dim_geography_city),
    ('dim_geography',         dim_geography),
    ('dim_products_detail',   dim_products_detail),
    ('dim_products',          dim_products),
    ('dim_shipment',          dim_shipment),
    ('dim_time',              dim_time),
    ('fact_orders',           fact_orders),
]

for nombre, tabla in tablas:
    tabla.to_sql(nombre, conn, index=False, if_exists='replace')

conn.commit()
conn.close()
print(f"superstore_model.db exportado ({len(tablas)} tablas)")

superstore_model.db exportado (13 tablas)


## 7. Validación final

In [12]:
conn = sqlite3.connect(db_path)
cur = conn.cursor()

checks = [
    ("Total Sales",       "SELECT ROUND(SUM(Sales),2) FROM fact_orders",                      2296919.49),
    ("Total Profit",      "SELECT ROUND(SUM(Profit),2) FROM fact_orders",                     286409.08),
    ("Profit Margin %",   "SELECT ROUND(SUM(Profit)/SUM(Sales)*100,2) FROM fact_orders",      12.47),
    ("Total Customers",   "SELECT COUNT(*) FROM dim_customers",                                793),
    ("Total Registros",   "SELECT COUNT(*) FROM fact_orders",                                  9993),
    ("Avg Ticket",        "SELECT ROUND(AVG(Sales),2) FROM fact_orders",                       229.85),
    ("Avg Discount %",    "SELECT ROUND(AVG(Discount)*100,2) FROM fact_orders",               15.62),
    ("Órdenes Pérdida",   "SELECT COUNT(*) FROM fact_orders WHERE Profit < 0",                1870),
]

resultados = []
for name, q, expected in checks:
    result = cur.execute(q).fetchone()[0]
    ok = abs(float(result) - float(expected)) < 0.02
    resultados.append({'KPI': name, 'Resultado': result, 'Esperado': expected, 'OK': ok})

conn.close()

df_val = pd.DataFrame(resultados).set_index('KPI')
all_ok = df_val['OK'].all()
print(f"{'Todos los KPIs validados' if all_ok else 'HAY DISCREPANCIAS'}")
df_val

Todos los KPIs validados


,Resultado,Esperado,OK
KPI,,,
Total Sales,2296919.49,2296919.49,True
Total Profit,286409.08,286409.08,True
Profit Margin %,12.47,12.47,True
Total Customers,793.00,793.00,True
Total Registros,9993.00,9993.00,True
Avg Ticket,229.85,229.85,True
Avg Discount %,15.62,15.62,True
Órdenes Pérdida,1870.00,1870.00,True
